# 01 / Short-Term Memory and the Context Window

**Deep Learning Indaba 2026 / Introduction to Agentic AI**

In notebook `00` we kept a running list of messages. Here we open that up: we manage the
story history **by hand**, watch it grow, and hit the wall every conversational agent hits,
the **context window**. Then we apply the simplest fix, a "story so far" summary.

**No API key:** we run the same small open model locally.

## 1.1 / Setup

In [ ]:
%pip install -q transformers accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")

def chat(messages, max_new_tokens=256, temperature=0.8):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    if temperature and temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)                  # greedy decoding when temperature is 0
    out = model.generate(**inputs, **gen)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
print("Ready on", model.device)

## 1.2 / Why models forget

A language model is **stateless**: each call is independent. It only knows what is inside
the request. So to continue a story we resend the **whole tale so far** every turn. We build
that loop ourselves with a plain list.

In [ ]:
history = [
    {"role": "system", "content": "You are a griot telling one continuous folktale. Keep each reply to one short paragraph and stay consistent."},
]

def tell(user_text):
    history.append({"role": "user", "content": user_text})
    reply = chat(history)
    history.append({"role": "assistant", "content": reply})
    return reply

print(tell("Start a tale: a girl named Zola in a village by the great river finds a talking drum."))

In [ ]:
# We never repeat the names, but because we resend the history, they persist.
print(tell("The drum gives her a warning. What does it say?"))

In [ ]:
print(tell("Remind me, what is the girl's name and what did she find?"))

## 1.3 / Watching the window fill up

A model can only read so many **tokens** at once: its *context window*. Our history grows
every turn, and a long saga eventually exceeds it. Because we loaded the tokenizer, we can
count the real token cost (no guessing).

In [ ]:
def token_count(messages):
    ids = tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
    return len(ids)

beats = [
    "She follows the drum into the forest.",
    "She meets a tortoise who asks three riddles.",
    "She answers the first riddle.",
    "A storm rises and she shelters in a baobab tree.",
    "At dawn she discovers a hidden village.",
]
for b in beats:
    tell(b)

print("Messages stored:", len(history))
print("Tokens we now resend every turn:", token_count(history))
print("Model context window:", tok.model_max_length, "tokens")
print("This only grows. On a long session or a big document, it eventually overflows.")

Even when the window is large, **you pay for every token you resend** (in time, and on
a hosted model in money too). Where bandwidth and budget are tight, resending a giant
history is a real cost, so we manage it.

## 1.4 / The simplest fix: a "story so far" summary

Instead of carrying every word forever, we **compress** the older turns into a short summary
and keep only the last few verbatim. A common, effective memory strategy.

In [ ]:
def summarise(messages):
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in messages if m['role'] != 'system')
    return chat([
        {"role": "system", "content": "Summarise the story so far in 3 short bullet points: characters, key objects, and what has happened."},
        {"role": "user", "content": transcript},
    ], max_new_tokens=256, temperature=0.3)

def compress_history(keep_last=4):
    global history
    system = history[0]
    body = history[1:]
    if len(body) <= keep_last:
        return None
    old, recent = body[:-keep_last], body[-keep_last:]
    summary = summarise(old)
    history = [system, {"role": "user", "content": "[Story so far]\n" + summary}] + recent
    return summary

before = token_count(history)
summary = compress_history(keep_last=4)
after = token_count(history)
print("STORY SO FAR:\n", summary)
print(f"\nTokens before: {before}  ->  after: {after}  (saved about {before - after})")

In [ ]:
# The griot still remembers the essentials from the compressed summary.
print(tell("Using everything so far, who is the heroine and what is she carrying?"))

## 1.5 / What we learned, and what is still missing

- An agent remembers by **resending its history**; there is no hidden state.
- That history is bounded by the **context window** and **costs tokens**, so we manage it,
  by dropping or, better, **summarising**.
- But this is still only short-term. Restart the kernel and `history` is empty; the whole
  saga is lost. Summarising also cannot pick out the one relevant past event from a long,
  branching tale.

For that we need **persistent, searchable long-term memory**: store every event and retrieve
only the relevant ones with **vector embeddings**. That is notebook `02`.